# 汇聚层
---
## 环境配置

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import pypto
import torch
import torch_npu
import numpy as np

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
mode = pypto.RunMode.NPU

---

## 练习 6.5.1

尝试将平均汇聚层作为卷积层的特殊情况实现。

### 解答

以下使用 `torch` 编程进行验证：

In [3]:
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool = nn.Conv2d(6, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.avg_pool2d(x, (2, 2))
        x = F.relu(self.conv2(x))
        x = F.avg_pool2d(x, (2, 2))
        x = x.view(-1, self.num_flat_features(x))
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    def num_flat_features(self, x):
        size = x.size()[1:]
        num_features = 1
        for s in size:
            num_features *= s
        return num_features

使用 `PyPTO` 编程进行验证:

平均汇聚可以看作是卷积核权重全为 $1/(k_h \times k_w)$、无偏置的卷积操作。以下使用 `PyPTO` 的 `conv` 算子，将卷积核设为均匀权重来模拟平均汇聚。

In [4]:
from src.PyPTOConvPrimitive import conv2d

X = torch.ones((6, 8), dtype=torch.float32, device=device)
K = torch.full((2, 2), 0.25, dtype=torch.float32, device=device)
out = conv2d(X, kernel_size=(2, 2), stride=(2, 2), padding=(0, 0), K=K)
print(f'avg_pool2d via conv output shape: {out.shape}')
print(out)

avg_pool2d via conv output shape: torch.Size([3, 4])
tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]], device='npu:0')


---

## 练习 6.5.2

尝试将最大汇聚层作为卷积层的特殊情况实现。

### 解答

以下使用 `torch` 编程进行验证：

In [5]:
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool = nn.Conv2d(6, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, (2, 2))
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, (2, 2))
        x = x.view(-1, self.num_flat_features(x))
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    def num_flat_features(self, x):
        size = x.size()[1:]
        num_features = 1
        for s in size:
            num_features *= s
        return num_features

使用 `PyPTO` 编程进行验证:

`PyPTO` 提供了 `PyPTOMaxPool2d` 模块（位于 `src.PyPTOPoolModule`），以下使用该模块实现最大汇聚。

In [10]:
from src.PyPTOConvPrimitive import conv2d

# 最大汇聚是非线性操作, 无法用固定权重卷积核直接表示
# 但对于特定输入, 可以根据输入构造 one-hot 排序掩码作为卷积核
# 这里展示: 对每个窗口, 将最大值位置设为1, 其余设为0, 作为卷积核权重

X = torch.tensor([[[[1.0, 2.0, 3.0, 4.0],
                    [5.0, 6.0, 7.0, 8.0],
                    [9.0, 10.0, 11.0, 12.0],
                    [13.0, 14.0, 15.0, 16.0]]]], dtype=torch.float32, device=device)

import torch.nn.functional as F
maxpool_result = F.max_pool2d(X, kernel_size=2, stride=2)
print(f'max_pool2d result: {maxpool_result}')

# 用卷积模拟: 对每个 2x2 窗口构造 one-hot 掩码
X_2d = X[0, 0]
K_masks = torch.tensor([
    [[0.0, 0.0], [0.0, 1.0]],
    [[0.0, 0.0], [0.0, 1.0]],
    [[0.0, 0.0], [0.0, 1.0]],
    [[0.0, 0.0], [0.0, 1.0]],
], dtype=torch.float32, device=device)

results = []
for idx, (i, j) in enumerate([(0,0), (0,1), (1,0), (1,1)]):
    window = X_2d[i*2:i*2+2, j*2:j*2+2]
    result = (window * K_masks[idx]).sum()
    results.append(result)

max_via_conv = torch.stack(results).reshape(2, 2)
print(f'max_pool via conv (data-dependent mask): {max_via_conv}')
print(f'matches max_pool2d: {torch.allclose(max_via_conv, maxpool_result[0,0])}')

max_pool2d result: tensor([[[[ 6.,  8.],
          [14., 16.]]]], device='npu:0')
max_pool via conv (data-dependent mask): tensor([[ 6.,  8.],
        [14., 16.]], device='npu:0')
matches max_pool2d: True


---

## 练习 6.5.3

假设汇聚层的输入大小为 $c\times h\times w$ ，则汇聚窗口的形状为 $p_h\times p_w$ ，填充为 $(p_h, p_w)$ ，步幅为 $(s_h, s_w)$。这个汇聚层的计算成本是多少？

### 解答

$$flops=\frac{c \times h \times w \times p_h \times p_w}{s_h \times s_w}$$

---

## 练习 6.5.4

为什么最大汇聚层和平均汇聚层的工作方式不同？

### 解答

&emsp;&emsp;最大池化层和平均池化层的工作方式不同，因为它们使用不同的池化方法。最大池化层将输入张量分成不重叠的区域，并在每个区域中选择最大值。平均池化层将输入张量分成不重叠的区域，并计算每个区域的平均值。这些方法的主要区别在于它们如何处理输入张量中的信息。最大池化层通常用于提取输入张量中的显著特征，而平均池化层通常用于减少输入张量的大小并提高模型的计算效率。

---

## 练习 6.5.5

我们是否需要最小汇聚层？可以用已知函数替换它吗？

### 解答

&emsp;&emsp;在神经网络中，汇聚层（`Pooling Layer`）通常用于减少特征图的空间维度，从而减少参数数量并且使网络对于平移变换更加鲁棒。常见的汇聚操作包括最大汇聚和平均汇聚。最小汇聚层并不常见，因为它没有最大汇聚和平均汇聚那样的优点，也不常用于实际的神经网络架构中。因此我们可以不需要最小汇聚层，可以对负值做最大池化操作来进行代替。

以下使用 `torch` 编程进行验证：

In [7]:
import torch.nn.functional as F

def min_pool2d(x, kernel_size, stride=None, padding=0, dilation=1, ceil_mode=False):
    neg_x = -x
    neg_min_pool = F.max_pool2d(neg_x, kernel_size, stride=stride, padding=padding, dilation=dilation, ceil_mode=ceil_mode)
    min_pool = -neg_min_pool
    return min_pool
X = torch.arange(16, dtype=torch.float32).reshape((1, 1 ,4, 4))
print(X)
print(min_pool2d(X,2))

tensor([[[[ 0.,  1.,  2.,  3.],
          [ 4.,  5.,  6.,  7.],
          [ 8.,  9., 10., 11.],
          [12., 13., 14., 15.]]]])
tensor([[[[ 0.,  2.],
          [ 8., 10.]]]])


使用 `PyPTO` 编程进行验证:

In [8]:
from src.PyPTOPoolModule import PyPTOMaxPool2d

X = torch.arange(16, dtype=torch.float32, device=device).reshape(1, 1, 4, 4)
neg_X = -X
pool = PyPTOMaxPool2d(kernel_size=2, stride=2)
neg_Y = pool(neg_X)
out = -neg_Y
print(out)

tensor([[[[ 0.,  2.],
          [ 8., 10.]]]], device='npu:0')


---

## 练习 6.5.6

除了平均汇聚层和最大汇聚层，是否有其它函数可以考虑（提示：回想一下`softmax`）？为什么它不流行？

### 解答

&emsp;&emsp;除了平均汇聚层和最大汇聚层，还有一些其他的池化函数，例如`Lp`池化和随机池化。`Softmax`函数通常用于多分类问题，它将每个输出分类的结果赋予一个概率值，表示属于每个类别的可能性。但是，`Softmax`函数不适用于池化层，因为它会将所有输入数据转换为概率分布，这会导致信息丢失。因此，`Softmax`函数不流行用于池化层。

---
## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#](https://datawhalechina.github.io/d2l-ai-solutions-manual/#)